# Step 4 — Multi-Agent

🇬🇧 **English** (this notebook)

Add a second agent with a different, complementary role. This notebook is **standalone**: two agents, chained by passing the first agent's answer into the second agent's question — no `Crew`, no `tasks.yaml`, no `context:` field needed. This is multi-agent in the simplest sense: role specialization with output chaining, not dynamic delegation.

## Background

A single agent can do multiple things, but it can't hold two genuinely different epistemic roles at the same time — it can't be simultaneously credulous (collecting everything) and skeptical (questioning what it found). Two agents let you encode that tension into the architecture. The seminal demonstration of agents collaborating through conversation was:

> Wu, Q., Bansal, G., Zhang, J., Wu, Y., Li, B., Zhu, E., Jiang, L., Zhang, X., Zhang, S., Liu, J., Awadallah, A. H., White, R. W., Burger, D., & Wang, C. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)

![AutoGen: conversable agents solving tasks in joint or hierarchical patterns](../assets/autogen-wu2023-fig1.png)
*Figure 1 from Wu et al. (2023). Reproduced for educational use in this course.*

## The exercise

Two agents, two `kickoff()` calls: the first agent's `result.raw` is handed to the second agent as part of its question — the same idea as step 2c's chain prompting, but each call now has its own role/goal/backstory instead of being anonymous.

This example reuses step 3's Researcher unchanged, and adds a new **Analyst** role — the same pairing this repo's full demo crew uses — that turns the Researcher's findings into a report aimed at one specific audience (a product team shipping an AI-powered hiring tool), a job the Researcher alone was never asked to do.

In [ ]:
import os

from dotenv import load_dotenv
from crewai import Agent

load_dotenv()

topic = "EU AI Act"

# ── Agent 1: Researcher — same "researcher" template as agents.yaml ──────────
agent_1_role      = f"{topic} Senior Data Researcher"
agent_1_goal      = f"Uncover cutting-edge developments in {topic}"
agent_1_backstory = (
    f"You're a seasoned researcher with a knack for uncovering the latest "
    f"developments in {topic}. Known for your ability to find the most relevant "
    f"information and present it in a clear and concise manner."
)

agent_1 = Agent(
    role=agent_1_role,
    goal=agent_1_goal,
    backstory=agent_1_backstory,
    verbose=True,
)

question_1 = (
    "Explain the EU AI Act's risk-based categories (unacceptable, high-risk, "
    "limited, minimal) and what obligations apply to providers of high-risk AI systems."
)
result_1 = agent_1.kickoff(question_1)
print("=== Agent 1 (Researcher) output ===")
print(result_1.raw)

# ── Agent 2: Analyst — same "analyst" template as agents.yaml ─────────────────
agent_2_role      = f"{topic} Reporting Analyst"
agent_2_goal      = f"Create detailed reports based on {topic} data analysis and research findings"
agent_2_backstory = (
    "You're a meticulous analyst with a keen eye for detail. You're known for "
    "your ability to turn complex data into clear and concise reports, making "
    "it easy for others to understand and act on the information you provide."
)

agent_2 = Agent(
    role=agent_2_role,
    goal=agent_2_goal,
    backstory=agent_2_backstory,
    verbose=True,
)

question_2 = (
    "Using the research below, write a short report for a product team that ships "
    "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
    "to them and by when.\n\n" + result_1.raw
)
result_2 = agent_2.kickoff(question_2)
print("\n=== Agent 2 (Analyst) output ===")
print(result_2.raw)

## Your task

1. Run the cell. Compare the Analyst's report to step 3's Researcher answer alone — is it more targeted and actionable, or does it mostly repackage the same content?

2. **Experiment**: change `question_2` to ignore `result_1.raw` (ask Agent 2 the product-team question directly, with no chaining). What's missing from Agent 2's answer once it can no longer see Agent 1's work?

3. Now swap in your own team's topic — keep the Researcher agent from step 3, and give it a second, genuinely complementary role suited to your topic.

4. Fill in the **Step 4** section of `EVALUATION.md`.

## Stretch goal

Add a third agent whose absence you'd actually notice — a critic, a translator for a non-expert audience, or a validator — by chaining `result_2.raw` into a third `kickoff()` call. Does the output meaningfully change? If it doesn't, ask yourself why.